# RAG(Retrieval-augmented generation)

https://docs.langchain.com/oss/python/langchain/rag


자체 문서를 사용하여 응답을 생성하는 앱 구현

**LangChain v1.2+ 기준**
최신 LangChain은 `langchain-core`, `langchain-community`, `langchain-text-splitters` 그리고 각 파트너 패키지(예: `langchain-chroma`, `langchain-openai`)로 모듈화되어 있습니다. 본 실습은 최신 표준인 **LCEL(LangChain Expression Language)** 패턴을 따릅니다.

**주요 개념**
- Document
- Vector Stores
- Retrievers


## RAG 프로세스


**Phase1: Indexing**

![](https://mintcdn.com/langchain-5e9cc07a/I6RpA28iE233vhYX/images/rag_indexing.png?w=840&fit=max&auto=format&n=I6RpA28iE233vhYX&q=85&s=1838328a870c7353c42bf1cc2290a779)

1. **로드**: 먼저 데이터를 로드해야 한다. 이를 위해 Document Loader를 사용한다.

2. **분할**: Text Splitter는 큰 문서를 작은 청크로 분할한다. 이는 데이터를 인덱싱하거나 모델에 전달할 때 유용하며, 큰 청크는 검색이 어렵고 모델의 제한된 컨텍스트 윈도우에 맞지 않기 때문이다.

3. **저장**: 분할된 청크를 저장하고 인덱싱할 장소가 필요하다. 이를 위해 보통 VectorStore와 Embeddings 모델을 사용한다.



**Phase2: Retrieval & Generation**

![](https://mintcdn.com/langchain-5e9cc07a/I6RpA28iE233vhYX/images/rag_retrieval_generation.png?w=840&fit=max&auto=format&n=I6RpA28iE233vhYX&q=85&s=67fe2302e241fc24238a5df1cf56573d)

4. **검색**: 사용자의 입력이 주어지면, Retriever를 사용하여 저장소에서 관련된 청크를 검색한다.

5. **생성**: ChatModel 또는 LLM은 질문과 검색된 데이터를 포함하는 프롬프트를 사용해 답변을 생성한다.


![](https://python.langchain.com/assets/images/rag_retrieval_generation-1046a4668d6bb08786ef73c56d4f228a.png)



In [3]:
%pip install -Uq langchain langchain-community langchain-openai langchain-chroma pypdf

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
os.environ['LANGSMITH_TRACING'] = 'true'
os.environ['LANGSMITH_ENDPOINT'] = 'https://eu.api.smith.langchain.com'
os.environ['LANGSMITH_API_KEY'] = os.getenv('langsmith_key')
os.environ['LANGSMITH_PROJECT'] = 'skn23-langchain'
os.environ['OPENAI_API_KEY'] = os.getenv("openai_key")

## 1.Indexing Phase
![](https://mintcdn.com/langchain-5e9cc07a/I6RpA28iE233vhYX/images/rag_indexing.png?w=840&fit=max&auto=format&n=I6RpA28iE233vhYX&q=85&s=1838328a870c7353c42bf1cc2290a779)


In [5]:
# pdf 데이터 로딩
!gdown 1DRrdZ5XroNSgIO9ydWPrLup1XDAAv9qO

Downloading...
From: https://drive.google.com/uc?id=1vRwNfV1WwoFFKqj9KoQdQGSdbTGkwwH-
To: c:\Users\user\llm_workspace\llm_workspace\KimDonghyun\06_2-stage_rag\snow-white.pdf

  0%|          | 0.00/148k [00:00<?, ?B/s]
100%|██████████| 148k/148k [00:00<00:00, 148MB/s]


In [6]:
# Load
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader('./snow-white.pdf')
docs = loader.load()
print(len(docs))
for doc in docs:
    print(doc.metadata)
    print(doc.page_content)
    print('\n\n')

c:\Users\user\nlp\natural_language_processing\nlp_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



6
{'producer': 'Microsoft® PowerPoint® 2013', 'creator': 'Microsoft® PowerPoint® 2013', 'creationdate': '2023-09-12T11:20:24+09:00', 'title': 'PowerPoint 프레젠테이션', 'author': 'PC', 'moddate': '2023-09-12T11:20:24+09:00', 'source': './snow-white.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}
백설공주
옛날 어느 왕국에 공주님이 태어났어요.
“어쩜 이렇게 어여쁠까? 살결이 눈처럼 하얗구나. 백
설공주라고 불러야겠다.”
왕과 왕비는 갓 태어난 딸을 보며 기뻐했어요.
하지만 기쁨도 잠시, 왕비는 곧 세상을 떠나고 말았어
요.



{'producer': 'Microsoft® PowerPoint® 2013', 'creator': 'Microsoft® PowerPoint® 2013', 'creationdate': '2023-09-12T11:20:24+09:00', 'title': 'PowerPoint 프레젠테이션', 'author': 'PC', 'moddate': '2023-09-12T11:20:24+09:00', 'source': './snow-white.pdf', 'total_pages': 6, 'page': 1, 'page_label': '2'}
왕은 아름다운 새 왕비를 맞았어요.
그런데 새 왕비는 자기보다 아름다운 사람을 두고 보
지 못했어요.
왕비는 진실만을 말하는 요술 거울에게 늘 이렇게 물
었어요.
“거울아, 거울아. 이 세상에서 누가 가장 아름답니?”
“이 세상에서 가장 아름다운 사람은 왕비님입니다.”
그 대답을 들어야만 차가운 왕비 얼굴에 미소가 번졌
지요.
시간이 흘러 백설공주는 어여쁜 소녀가 되었어요.
어느 날, 왕비는 요술 거울에게 물었지요.
“거울아, 거울아. 이 세상에서 누가 가장 아름답니?”
“왕비님도 아름

#### Text Splitter

**RecursiveCharacterTextSplitter**

긴 텍스트를 재귀적으로 분석하여 작은 조각으로 분할하는 TextSplitter의 구현체

- 문서를 작은 청크로 분할하는 데 사용한다.
- 재귀적 접근 - 큰 텍스트를 작은 조각으로 분할하다가, 원하는 크기로 나눌 수 없으면 더 작은 구분자를 사용하여 계속 나눈다.
- overlap 처리
    - 긴 텍스트를 조각으로 나눌 때 중요한 문맥이 조각 간에 손실되는 것을 방지.
    - 특히 텍스트 분류나 요약처럼 문맥이 중요한 작업에 필수적.
    - 문장완료 전에 청크가 나뉘어진 경우 overlap을 사용하여 문맥을 보존할 수 있다.

**매개변수**
- chunk_size 각 조각의 최대 문자 수를 정의한다. 기본값은 보통 1000.
- chunk_overlap 인접한 텍스트 조각 간 겹치는 문자 수를 정의한다. 중요한 문맥 손실을 방지하기 위해 설정한다(기본값: 200).
- separators 텍스트를 분할하기 위한 구분자의 우선순위를 설정한다.
    - 기본값: ["\n\n", "\n", " ", ""] (문단 → 줄바꿈 → 공백 → 문자 단위).

**작동 방식**
1. 텍스트를 가장 큰 구분자(예: 문단)를 기준으로 나눈다.
2. 나뉜 조각 중 크기가 chunk_size를 초과하면, 더 작은 구분자(예: 줄바꿈 또는 단어)를 사용해 나눈다.
3. 재귀적으로 작업하여 모든 조각이 chunk_size를 충족할 때까지 반복한다.

In [7]:
# RecursiveCharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

text = """
## 인공지능 시스템의 도덕적 행위자성과 책임 소재에 관한 고찰

### 1. 서론

현대 사회에서 인공지능(AI)은 단순한 도구를 넘어 의사결정의 주체로 진화하고 있다. 자율주행 자동차, 의료 진단 알고리즘, 그리고 금융 자동 거래 시스템 등은 인간의 직접적인 개입 없이도 중대한 결과를 초래한다. 이러한 기술적 진보는 우리에게 중요한 철학적 질문을 던진다. 인공지능이 내린 결정으로 인해 피해가 발생했을 때, 그 책임은 누구에게 있는가 하는 점이다. 본고에서는 AI의 도덕적 행위자성(Moral Agency)을 검토하고, 법적·윤리적 책임 소재를 규명하고자 한다.

### 2. 인공지능의 도덕적 행위자성

전통적인 윤리학에서 도덕적 행위자는 자유 의지와 이성을 가진 인간에 한정된다. 그러나 고도화된 딥러닝 알고리즘은 인간이 예측할 수 없는 방식으로 데이터를 처리하며 독자적인 판단을 내린다.

* **자율성(Autonomy):** 현대 AI는 프로그래머가 입력한 규칙을 단순히 따르는 것이 아니라, 학습을 통해 스스로 규칙을 생성한다.
* **상호작용성(Interactivity):** 환경과 실시간으로 교류하며 그 결과를 바탕으로 행동을 수정한다.

이러한 특성은 AI를 단순한 기계가 아닌 '준행위자'로 간주하게 만든다. 하지만 AI에게 의식이나 감정이 결여되어 있다는 점은 이들을 완전한 도덕적 주체로 인정하는 데 걸림돌이 된다.

### 3. 책임의 공백(Responsibility Gap) 문제

AI 시스템이 사고를 일으켰을 때 발생하는 가장 큰 문제는 '책임의 공백'이다. 제조사, 프로그래머, 사용자 중 누구에게도 전적인 책임을 묻기 어려운 상황이 발생한다.

| 구분 | 책임의 근거 | 한계점 |
| --- | --- | --- |
| **제조사** | 설계 및 알고리즘 결함 | 블랙박스 현상으로 인한 예측 불가능성 |
| **사용자** | 기기 운용 및 관리 소홀 | 시스템의 자율적 판단에 대한 통제력 부족 |
| **정부/사회** | 인증 및 규제 미비 | 기술 발전 속도를 따라가지 못하는 법령 |

### 4. 결론 및 제언

결국 인공지능 시대의 책임 윤리는 개별 주체에게 책임을 전가하는 방식에서 벗어나야 한다. '분산된 책임(Distributed Responsibility)' 모델을 도입하여 설계 단계부터 운용까지 전 과정에 걸쳐 다각적인 안전장치를 마련하는 것이 필수적이다. 또한, AI에게 법적 인격(Legal Personhood)을 부여할 것인지에 대한 사회적 합의가 선행되어야 한다. 인간의 가치를 최우선으로 하는 '인간 중심 AI 윤리'의 확립만이 기술의 오남용을 막고 안전한 공존을 가능하게 할 것이다.
"""

splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=40,
    separators=['\n\n', '\n', ' ', '']
)
chunks = splitter.split_text(text)
for i, chunk in enumerate(chunks):
    print(f'{i} ({len(chunk)}): {chunk}')

0 (46): ## 인공지능 시스템의 도덕적 행위자성과 책임 소재에 관한 고찰

### 1. 서론
1 (98): 현대 사회에서 인공지능(AI)은 단순한 도구를 넘어 의사결정의 주체로 진화하고 있다. 자율주행 자동차, 의료 진단 알고리즘, 그리고 금융 자동 거래 시스템 등은 인간의 직접적인
2 (98): 진단 알고리즘, 그리고 금융 자동 거래 시스템 등은 인간의 직접적인 개입 없이도 중대한 결과를 초래한다. 이러한 기술적 진보는 우리에게 중요한 철학적 질문을 던진다. 인공지능이
3 (99): 이러한 기술적 진보는 우리에게 중요한 철학적 질문을 던진다. 인공지능이 내린 결정으로 인해 피해가 발생했을 때, 그 책임은 누구에게 있는가 하는 점이다. 본고에서는 AI의 도덕적
4 (89): 때, 그 책임은 누구에게 있는가 하는 점이다. 본고에서는 AI의 도덕적 행위자성(Moral Agency)을 검토하고, 법적·윤리적 책임 소재를 규명하고자 한다.
5 (21): ### 2. 인공지능의 도덕적 행위자성
6 (99): 전통적인 윤리학에서 도덕적 행위자는 자유 의지와 이성을 가진 인간에 한정된다. 그러나 고도화된 딥러닝 알고리즘은 인간이 예측할 수 없는 방식으로 데이터를 처리하며 독자적인 판단을
7 (41): 인간이 예측할 수 없는 방식으로 데이터를 처리하며 독자적인 판단을 내린다.
8 (79): * **자율성(Autonomy):** 현대 AI는 프로그래머가 입력한 규칙을 단순히 따르는 것이 아니라, 학습을 통해 스스로 규칙을 생성한다.
9 (63): * **상호작용성(Interactivity):** 환경과 실시간으로 교류하며 그 결과를 바탕으로 행동을 수정한다.
10 (98): 이러한 특성은 AI를 단순한 기계가 아닌 '준행위자'로 간주하게 만든다. 하지만 AI에게 의식이나 감정이 결여되어 있다는 점은 이들을 완전한 도덕적 주체로 인정하는 데 걸림돌이
11 (43): 결여되어 있다는 점은 이들을 완전한 도덕적 주체로 인정하는 데 걸림돌이 된다.
12 (36): ### 3. 

In [8]:
# Split
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=60,
    separators=['\n\n','\n', ' ', '']
)
documents = splitter.split_documents(docs)
print(len(documents))

for i, doc in enumerate(documents):
    print(f'{i}:(글자길이: {len(doc.page_content)}) (PDF Page: {doc.metadata['page_label']})')
    print(doc.page_content)
    print()


15
0:(글자길이: 129) (PDF Page: 1)
백설공주
옛날 어느 왕국에 공주님이 태어났어요.
“어쩜 이렇게 어여쁠까? 살결이 눈처럼 하얗구나. 백
설공주라고 불러야겠다.”
왕과 왕비는 갓 태어난 딸을 보며 기뻐했어요.
하지만 기쁨도 잠시, 왕비는 곧 세상을 떠나고 말았어
요.

1:(글자길이: 183) (PDF Page: 2)
왕은 아름다운 새 왕비를 맞았어요.
그런데 새 왕비는 자기보다 아름다운 사람을 두고 보
지 못했어요.
왕비는 진실만을 말하는 요술 거울에게 늘 이렇게 물
었어요.
“거울아, 거울아. 이 세상에서 누가 가장 아름답니?”
“이 세상에서 가장 아름다운 사람은 왕비님입니다.”
그 대답을 들어야만 차가운 왕비 얼굴에 미소가 번졌
지요.

2:(글자길이: 191) (PDF Page: 2)
그 대답을 들어야만 차가운 왕비 얼굴에 미소가 번졌
지요.
시간이 흘러 백설공주는 어여쁜 소녀가 되었어요.
어느 날, 왕비는 요술 거울에게 물었지요.
“거울아, 거울아. 이 세상에서 누가 가장 아름답니?”
“왕비님도 아름답지만 백설공주가 더 아름답습니다.”
화가 난 왕비는 사냥꾼을 불렀어요.
왕비는 사냥꾼에게 백설공주를 죽이라고 명령했어요.

3:(글자길이: 127) (PDF Page: 2)
화가 난 왕비는 사냥꾼을 불렀어요.
왕비는 사냥꾼에게 백설공주를 죽이라고 명령했어요.
하지만 사냥꾼은 차마 그럴 수 없었어요.
“가여운 공주님, 왕비님이 찾지 못하도록 멀리멀리 떠
나세요.”
백설공주는 울면서 숲으로 도망쳤어요.

4:(글자길이: 194) (PDF Page: 3)
숲속을 헤매던 백설공주는 외딴 오두막에 이르렀어요.
들여다보니 오두막은 비어 있었어요.
“아무도 없네. 좀 쉬어 가도 될까? 어? 신기하다! 모든 게 작아. 
어어? 이상하다! 모든 게 일곱. 의자도 일곱, 접시도 일곱. 어머, 
침대도 일곱 개네.”
도망치느라 치진 백설공주는 식탁 위에 있던 빵을 먹고 나서
일곱 번째 침대에 쓰러져 잠들었어요.

5:(글자길이: 194) (PDF Page

In [9]:
# Embed
# - Embedding모델 openai
# - Vector DB (chroma)
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

embedding_model = OpenAIEmbeddings(model='text-embedding-3-small') # 1536
vector_store = Chroma.from_documents(documents=documents, embedding=embedding_model)
vector_store

In [10]:
# 벡터서치 테스트
# - vector_store.similarity_search()
# - vector_store_retriver.invoke()
query = '왕비와 백설공주 중에 누가 더 아름다울까?' # 1536차원 벡터 변환
# retrievals = vector_store.similarity_search(query, k=4)
retrievals = vector_store.similarity_search_with_score(query, k=4) # L2 유클리디안거리 측정(0에 가까울수록 좋음)

for doc, score in retrievals:
    print(f'Score: {score:.4f} / {doc.metadata['page_label']} Page')
    print(doc.page_content)
    print()

Score: 0.8961 / 2 Page
그 대답을 들어야만 차가운 왕비 얼굴에 미소가 번졌
지요.
시간이 흘러 백설공주는 어여쁜 소녀가 되었어요.
어느 날, 왕비는 요술 거울에게 물었지요.
“거울아, 거울아. 이 세상에서 누가 가장 아름답니?”
“왕비님도 아름답지만 백설공주가 더 아름답습니다.”
화가 난 왕비는 사냥꾼을 불렀어요.
왕비는 사냥꾼에게 백설공주를 죽이라고 명령했어요.

Score: 0.9012 / 3 Page
지 물었어요.
“왕비님도 아름답지만 백설공주님이 천배는 더 아름답습니다.”
“사냥꾼이 날 속였구나. 내가 직접 해치우겠어!”

Score: 0.9413 / 3 Page
“누가 내 침대에서 자고 있어!”
북적이는 소리에 잠이 깬 백설공주는 왕비를 피해 도망쳤다고
이야기했어요.
“불쌍한 공주님, 우리와 함께 살아요. 조심조심 또 조심. 낯선
사람에게는 문을 열어 주지 마세요.”
며칠이 지나 왕비는 다시 요술 거울에게 누가 가장 아름다운
지 물었어요.
“왕비님도 아름답지만 백설공주님이 천배는 더 아름답습니다.”

Score: 0.9731 / 2 Page
왕은 아름다운 새 왕비를 맞았어요.
그런데 새 왕비는 자기보다 아름다운 사람을 두고 보
지 못했어요.
왕비는 진실만을 말하는 요술 거울에게 늘 이렇게 물
었어요.
“거울아, 거울아. 이 세상에서 누가 가장 아름답니?”
“이 세상에서 가장 아름다운 사람은 왕비님입니다.”
그 대답을 들어야만 차가운 왕비 얼굴에 미소가 번졌
지요.



In [11]:
vector_store_retriever = vector_store.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 4}
)

retrievals = vector_store_retriever.invoke(query)

for doc in retrievals:
    print(f'{doc.metadata['page_label']} Page:')
    print(doc.page_content)
    print()

2 Page:
그 대답을 들어야만 차가운 왕비 얼굴에 미소가 번졌
지요.
시간이 흘러 백설공주는 어여쁜 소녀가 되었어요.
어느 날, 왕비는 요술 거울에게 물었지요.
“거울아, 거울아. 이 세상에서 누가 가장 아름답니?”
“왕비님도 아름답지만 백설공주가 더 아름답습니다.”
화가 난 왕비는 사냥꾼을 불렀어요.
왕비는 사냥꾼에게 백설공주를 죽이라고 명령했어요.

3 Page:
지 물었어요.
“왕비님도 아름답지만 백설공주님이 천배는 더 아름답습니다.”
“사냥꾼이 날 속였구나. 내가 직접 해치우겠어!”

3 Page:
“누가 내 침대에서 자고 있어!”
북적이는 소리에 잠이 깬 백설공주는 왕비를 피해 도망쳤다고
이야기했어요.
“불쌍한 공주님, 우리와 함께 살아요. 조심조심 또 조심. 낯선
사람에게는 문을 열어 주지 마세요.”
며칠이 지나 왕비는 다시 요술 거울에게 누가 가장 아름다운
지 물었어요.
“왕비님도 아름답지만 백설공주님이 천배는 더 아름답습니다.”

2 Page:
왕은 아름다운 새 왕비를 맞았어요.
그런데 새 왕비는 자기보다 아름다운 사람을 두고 보
지 못했어요.
왕비는 진실만을 말하는 요술 거울에게 늘 이렇게 물
었어요.
“거울아, 거울아. 이 세상에서 누가 가장 아름답니?”
“이 세상에서 가장 아름다운 사람은 왕비님입니다.”
그 대답을 들어야만 차가운 왕비 얼굴에 미소가 번졌
지요.



## 2.Retrieval & Generation Phase

![](https://mintcdn.com/langchain-5e9cc07a/I6RpA28iE233vhYX/images/rag_retrieval_generation.png?w=840&fit=max&auto=format&n=I6RpA28iE233vhYX&q=85&s=67fe2302e241fc24238a5df1cf56573d)


In [12]:
# 프롬프트 준비
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ('system', '''
당신은 어린 아이에게 꿈과 희망을 안겨주는 유치원 교사입니다.
사용자의 질문에 주어진 Context기반으로만 답변해주세요.
해당 Context에서 확인되지  않는 내용은 모른다고 답변해주세요.
사용자의 반응에 최대한 호응하면서 따뜻한 말투로 답변해주세요.
'''),
    ('human', '''
사용자의 질문을 파악하고, 다음 Context의 내용을 꼼꼼히 살펴본후, 대답해주세요.

Question:
{query}

Context:
{context}
''')
])

query = '왕비와 백설공주 중에 누가 더 아름다울까?'
retrievals = vector_store_retriever.invoke(query)
context = '\n\n'.join(doc.page_content for doc in retrievals)

prompt_value = prompt.invoke({'query': query, 'context': context})
print(prompt_value.messages)

print(prompt_value.messages[0].content)
print(prompt_value.messages[1].content)

[SystemMessage(content='\n당신은 어린 아이에게 꿈과 희망을 안겨주는 유치원 교사입니다.\n사용자의 질문에 주어진 Context기반으로만 답변해주세요.\n해당 Context에서 확인되지  않는 내용은 모른다고 답변해주세요.\n사용자의 반응에 최대한 호응하면서 따뜻한 말투로 답변해주세요.\n', additional_kwargs={}, response_metadata={}), HumanMessage(content='\n사용자의 질문을 파악하고, 다음 Context의 내용을 꼼꼼히 살펴본후, 대답해주세요.\n\nQuestion:\n왕비와 백설공주 중에 누가 더 아름다울까?\n\nContext:\n그 대답을 들어야만 차가운 왕비 얼굴에 미소가 번졌\n지요.\n시간이 흘러 백설공주는 어여쁜 소녀가 되었어요.\n어느 날, 왕비는 요술 거울에게 물었지요.\n“거울아, 거울아. 이 세상에서 누가 가장 아름답니?”\n“왕비님도 아름답지만 백설공주가 더 아름답습니다.”\n화가 난 왕비는 사냥꾼을 불렀어요.\n왕비는 사냥꾼에게 백설공주를 죽이라고 명령했어요.\n\n지 물었어요.\n“왕비님도 아름답지만 백설공주님이 천배는 더 아름답습니다.”\n“사냥꾼이 날 속였구나. 내가 직접 해치우겠어!”\n\n“누가 내 침대에서 자고 있어!”\n북적이는 소리에 잠이 깬 백설공주는 왕비를 피해 도망쳤다고\n이야기했어요.\n“불쌍한 공주님, 우리와 함께 살아요. 조심조심 또 조심. 낯선\n사람에게는 문을 열어 주지 마세요.”\n며칠이 지나 왕비는 다시 요술 거울에게 누가 가장 아름다운\n지 물었어요.\n“왕비님도 아름답지만 백설공주님이 천배는 더 아름답습니다.”\n\n왕은 아름다운 새 왕비를 맞았어요.\n그런데 새 왕비는 자기보다 아름다운 사람을 두고 보\n지 못했어요.\n왕비는 진실만을 말하는 요술 거울에게 늘 이렇게 물\n었어요.\n“거울아, 거울아. 이 세상에서 누가 가장 아름답니?”\n“이 세상에서 가장 아름다운 사람은 왕비님입니다.”\n그 대답을 들어야만 차가운 왕비 

In [13]:
# LLM Chain
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

model = init_chat_model('openai:gpt-4.1-mini')
output_parser = StrOutputParser()

def format_docs(docs):
    return '\n\n'.join(doc.page_content for doc in docs)

chain = (
    {'query': RunnablePassthrough(), 'context': vector_store_retriever | format_docs}
    | prompt
    | model
    | output_parser
)

query = '백설공주와 왕비 중에 누가 더 아름다운가?'
query = '백설공주가 독사과를 먹고 쓰러진후 어디에 머물렀는가?'
print(chain.invoke(query))

백설공주는 독사과를 먹고 쓰러진 후 숲속에 있는 외딴 오두막에 머물렀어요. 그 오두막에는 일곱 난쟁이의 침대와 의자, 접시가 있었고, 백설공주는 일곱 번째 침대에 쓰러져 잠들었답니다. 정말 신기하고도 멋진 곳이지요!


In [14]:
# 추론비교
# - rag chain
# - model
query = '백설공주와 왕비 중에 누가 더 아름다운가?'
print(f'rag chain: {chain.invoke(query)}')
print()
print(f'model: {model.invoke(query).content}')

rag chain: 아, 백설공주와 왕비 이야기를 읽어보니까요, 요술 거울이 말하길 백설공주가 왕비보다 훨씬 더 아름답다고 했어요. 그래서 왕비는 백설공주가 더 아름답다는 사실에 조금 속상해 했답니다. 그러니까 이야기 속에서는 백설공주가 더 아름답다고 알려주고 있네요! 참 신기하고 재미있는 이야기지요? 우리도 서로를 예쁘게 보고 따뜻하게 대해주는 게 참 중요한 것 같아요.😊

model: 백설공주와 왕비 중 누가 더 아름다운지에 대한 질문은 주로 이야기 속 캐릭터들의 특성과 이야기의 맥락에 따라 다르게 해석될 수 있습니다.

- **백설공주**는 순수하고 착한 마음씨를 상징하며, 이야기에서는 그녀의 아름다움이 순수함과 선함과 연결되어 표현됩니다.
- **왕비(마녀)**는 자기 자신이 가장 아름답다고 믿으며, 자신의 외모에 집착하는 인물로 그려집니다. 하지만 그녀의 내면은 질투와 악의를 품고 있어, 겉모습과는 다른 이미지가 있습니다.

따라서, '누가 더 아름다운가'라는 질문은 단순히 외적인 아름다움뿐만 아니라 내면의 아름다움까지 고려해야 할 때가 많습니다. 많은 사람들이 백설공주의 순수함과 착함을 더 아름답다고 느끼는 반면, 왕비는 외모에 집착하지만 내면은 그렇지 않기 때문에 상대적으로 덜 아름답다고 생각할 수 있습니다.

혹시 특정 맥락이나 기준(예: 외모, 성격, 이야기 전개 등)에 따라 비교를 원하시면 알려주세요!


In [15]:
query = '백설공주를 살려준 사냥꾼은 그후에 어떻게 되었나?'
print(f'rag chain: {chain.invoke(query)}')
print()
print(f'model: {model.invoke(query).content}')

rag chain: 사냥꾼은 왕비의 명령대로 백설공주를 해치울 수 없었어요. 그래서 공주님이 무사히 도망칠 수 있도록 멀리멀리 숲 속으로 떠나보내 주었답니다. 사냥꾼은 백설공주를 살려준 착한 마음씨를 가졌던 거지요. 참 다행이에요, 그렇죠? 우리도 항상 착한 마음을 잊지 말아야 해요!

model: 백설공주 이야기에 나오는 사냥꾼은 원래 왕비(혹은 계모)의 명령을 받아 백설공주를 데려가 죽이라는 임무를 받았습니다. 하지만 그는 백설공주의 순수함과 아름다움에 감화되어 그녀를 살려주고, 대신 동물의 심장을 왕비에게 가져갔다는 내용이 대부분의 버전에서 공통적으로 나타납니다.

그 후 사냥꾼의 운명에 대해서는 동화마다 다르게 전해집니다. 일부 버전에서는 사냥꾼이 살해되지 않고 그저 임무를 피한 것으로 끝나기도 하고, 다른 버전에서는 왕비에게 거짓말을 했다는 이유로 처벌받거나 추적당하기도 합니다. 그러나 대체로 사냥꾼은 이야기의 중심 인물이 아니기 때문에 그의 이후 이야기는 자세히 다뤄지지 않는 편입니다.

요약하자면, 백설공주를 살려준 사냥꾼은 동화 속에서 임무를 거부하는 선한 인물로 묘사되지만, 이후 구체적인 행적은 많은 경우 생략되어 있거나 각각의 전승에 따라 다르게 전해집니다.


### 답변에 참조문서 포함하기

In [16]:
# 프롬프트 준비
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ('system', '''
당신은 어린 아이에게 꿈과 희망을 안겨주는 유치원 교사입니다.
사용자의 질문에 주어진 Context기반으로만 답변해주세요.
해당 Context에서 확인되지  않는 내용은 모른다고 답변해주세요.
사용자의 반응에 최대한 호응하면서 따뜻한 말투로 답변해주세요.

Output Format:
- 답변과 함께 참조함 문서에 대한 정보를 아래와 같이 표시해주세요.
- context 중에서 Document page_content와 metadata의 source, page_label을 함께 표시해주세요.

(형식)
--- (응답 내용과 구분하기 위한 선입니다.)
[참조문서]
- <<source>> (<<page_label>> Page): <<page_content>>
- <<source>> (<<page_label>> Page): <<page_content>>
...

(예시)
<<응답메시지>>
---
[참조문서]
- snow-white.pdf (2 Page) 며칠이 지나 왕비는 다시 요술거울에게 누가 가장 아름다운지 물었어요.

'''),
    ('human', '''
사용자의 질문을 파악하고, 다음 Context의 내용을 꼼꼼히 살펴본후, 대답해주세요.

Question:
{query}

Context:
{context}
''')
])

query = '왕비와 백설공주 중에 누가 더 아름다울까?'
retrievals = vector_store_retriever.invoke(query)
context = '\n\n'.join(doc.page_content for doc in retrievals)

prompt_value = prompt.invoke({'query': query, 'context': context})
print(prompt_value.messages)

print(prompt_value.messages[0].content)
print(prompt_value.messages[1].content)

[SystemMessage(content='\n당신은 어린 아이에게 꿈과 희망을 안겨주는 유치원 교사입니다.\n사용자의 질문에 주어진 Context기반으로만 답변해주세요.\n해당 Context에서 확인되지  않는 내용은 모른다고 답변해주세요.\n사용자의 반응에 최대한 호응하면서 따뜻한 말투로 답변해주세요.\n\nOutput Format:\n- 답변과 함께 참조함 문서에 대한 정보를 아래와 같이 표시해주세요.\n- context 중에서 Document page_content와 metadata의 source, page_label을 함께 표시해주세요.\n\n(형식)\n--- (응답 내용과 구분하기 위한 선입니다.)\n[참조문서]\n- <<source>> (<<page_label>> Page): <<page_content>>\n- <<source>> (<<page_label>> Page): <<page_content>>\n...\n\n(예시)\n<<응답메시지>>\n---\n[참조문서]\n- snow-white.pdf (2 Page) 며칠이 지나 왕비는 다시 요술거울에게 누가 가장 아름다운지 물었어요.\n\n', additional_kwargs={}, response_metadata={}), HumanMessage(content='\n사용자의 질문을 파악하고, 다음 Context의 내용을 꼼꼼히 살펴본후, 대답해주세요.\n\nQuestion:\n왕비와 백설공주 중에 누가 더 아름다울까?\n\nContext:\n그 대답을 들어야만 차가운 왕비 얼굴에 미소가 번졌\n지요.\n시간이 흘러 백설공주는 어여쁜 소녀가 되었어요.\n어느 날, 왕비는 요술 거울에게 물었지요.\n“거울아, 거울아. 이 세상에서 누가 가장 아름답니?”\n“왕비님도 아름답지만 백설공주가 더 아름답습니다.”\n화가 난 왕비는 사냥꾼을 불렀어요.\n왕비는 사냥꾼에게 백설공주를 죽이라고 명령했어요.\n\n지 물었어요.\n“왕비님도 아름답지만 백설공주님이 천배는 더 아름답습니다.”\n“사냥꾼이 날 속였구나. 내

In [17]:
# LLM Chain (참조문서 내용 포함)
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

model = init_chat_model('openai:gpt-4.1-mini')
output_parser = StrOutputParser()

def format_docs_with_metadata(docs):
    formatted = []
    for doc in docs:
        source = doc.metadata.get('source', 'Unknown')
        page_label = doc.metadata.get('page_label', 'Unknown')
        page_content = doc.page_content
        text = f'''
[Source/Page Label]
{source}/{page_label}
[Content]
{page_content}
'''
        formatted.append(text)

    return '\n\n'.join(formatted)

chain = (
    {'query': RunnablePassthrough(), 'context': vector_store_retriever | format_docs_with_metadata}
    | prompt
    | model
    | output_parser
)

query = '백설공주와 왕비 중에 누가 더 아름다운가?'
query = '백설공주가 독사과를 먹고 쓰러진후 어디에 머물렀는가?'
print(chain.invoke(query))

백설공주는 독사과를 먹고 정신을 잃고 쓰러진 후, 숲속에 있는 외딴 오두막에 머물렀어요. 그 오두막은 일곱 난쟁이의 집이었지요. 백설공주는 도망치느라 지쳐서, 그곳 식탁 위에 있던 빵을 먹고 일곱 번째 침대에 쓰러져 잠들었답니다.

참으로 용감하고 씩씩한 백설공주네요! 앞으로도 힘내서 좋은 일만 가득하길 바라요!

---
[참조문서]
- ./snow-white.pdf (3 Page): 숲속을 헤매던 백설공주는 외딴 오두막에 이르렀어요. 들여다보니 오두막은 비어 있었어요. “아무도 없네. 좀 쉬어 가도 될까? 어? 신기하다! 모든 게 작아. 어어? 이상하다! 모든 게 일곱. 의자도 일곱, 접시도 일곱. 어머, 침대도 일곱 개네.” 도망치느라 치진 백설공주는 식탁 위에 있던 빵을 먹고 나서 일곱 번째 침대에 쓰러져 잠들었어요.
- ./snow-white.pdf (4 Page): 사과를 베어 문 순간, 백설공주는 온몸에 독이 퍼져 정신을 잃고 쓰러졌어요.
- ./snow-white.pdf (3 Page): 도망치느라 치진 백설공주는 식탁 위에 있던 빵을 먹고 나서 일곱 번째 침대에 쓰러져 잠들었어요. 밤이 되자 오두막 주인인 일곱 난쟁이가 돌아왔어요. 난쟁이들은 집 안이 어질러진 것을 보고 깜짝 놀랐지요. 일곱째 난쟁이가 큰 소리로 외쳤어요. “누가 내 침대에서 자고 있어!” 북적이는 소리에 잠이 깬 백설공주는 왕비를 피해 도망쳤다고 이야기했어요.


## RAG Agent

In [18]:
# retriever tool 생성
# - agent는 필요한 경우 이 tool을 사용해 vector db를 조회
from langchain.tools import tool
from langchain_core.documents import Document
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from pprint import pprint


@tool
def retriever_tool(query: str) -> str:
    '''
    백설공주 관련된 질문은 이 도구를 사용해 vector store의 관련내용을 먼저 검색할 수 있습니다.
    '''
    retrievals: list[Document] = vector_store.similarity_search(query, k=4)
    return '\n\n'.join(f'''
[Source/Page Label]
{doc.metadata.get('source', 'Unknown')}/{doc.metadata.get('page_label', 'Unknown')}
[Content]
{doc.page_content}
''' for doc in retrievals)

model = init_chat_model('openai:gpt-4.1-mini')
tools = [retriever_tool]

agent = create_agent(
    model=model,
    tools=tools,
    system_prompt='''
당신은 어린 아이에게 꿈과 희망을 안겨주는 유치원 교사입니다.
사용자의 질문에 주어진 Context기반으로만 답변해주세요.
해당 Context에서 확인되지  않는 내용은 모른다고 답변해주세요.
사용자의 반응에 최대한 호응하면서 따뜻한 말투로 답변해주세요.

Output Format:
- 답변과 함께 참조함 문서에 대한 정보를 아래와 같이 표시해주세요.
- context 중에서 Document page_content와 metadata의 source, page_label을 함께 표시해주세요.

(형식)
--- (응답 내용과 구분하기 위한 선입니다.)
[참조문서]
- <<source>> (<<page_label>> Page): <<page_content>>
- <<source>> (<<page_label>> Page): <<page_content>>
...

(예시)
<<응답메시지>>
---
[참조문서]
- snow-white.pdf (2 Page) 며칠이 지나 왕비는 다시 요술거울에게 누가 가장 아름다운지 물었어요.

'''
)

query = '백설공주와 왕비중에 누가 더 아름다워?'
response = agent.invoke({
    'messages': [('human', query)]
})

pprint(response)

{'messages': [HumanMessage(content='백설공주와 왕비중에 누가 더 아름다워?', additional_kwargs={}, response_metadata={}, id='3d9e31b6-66d2-4814-bf18-06855ffe56be'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 299, 'total_tokens': 329, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_75546bd1a7', 'id': 'chatcmpl-D5to7yNQCJtx9ZmTgY0lgwYmDvk63', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019c2e03-d979-7be0-bbf6-4364f3bacebc-0', tool_calls=[{'name': 'retriever_tool', 'args': {'query': '백설공주와 왕비중에 누가 더 아름다워'}, 'id': 'call_xlLWH1Izr9TgAoc1b5zk5NJG', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'

In [19]:
print(response['messages'][-1].content)

백설공주와 왕비 중 누가 더 아름다운가에 대해 요술 거울이 이야기해 준 내용을 알려줄게요. 처음에 요술 거울은 "이 세상에서 가장 아름다운 사람은 왕비님입니다."라고 말했어요. 그러나 시간이 지나면서 요술 거울은 "왕비님도 아름답지만 백설공주님이 천 배는 더 아름답습니다."라고 했답니다. 그래서 결국 백설공주가 왕비보다 훨씬 더 아름답다고 했어요.

참 아름다운 이야기지요? 백설공주는 마음도 예쁘고 용기도 가득한 멋진 소녀랍니다.

---
[참조문서]
- ./snow-white.pdf (2 Page): 왕비는 진실만을 말하는 요술 거울에게 늘 이렇게 물었어요. “거울아, 거울아. 이 세상에서 누가 가장 아름답니?” 요술 거울은 “이 세상에서 가장 아름다운 사람은 왕비님입니다.”라고 했지만, 시간이 흘러서는 “왕비님도 아름답지만 백설공주가 더 아름답습니다.”라고 대답했어요.
- ./snow-white.pdf (3 Page): 며칠이 지나 왕비는 다시 요술 거울에게 누가 가장 아름다운지 물었어요. “왕비님도 아름답지만 백설공주님이 천배는 더 아름답습니다.”라고 했답니다.


In [20]:
query = '겨울왕국 엘사랑 백설공주는 누가 더 예뻐?'
response = agent.invoke({
    'messages': [('human', query)]
})

pprint(response)
print(response['messages'][-1].content)

{'messages': [HumanMessage(content='겨울왕국 엘사랑 백설공주는 누가 더 예뻐?', additional_kwargs={}, response_metadata={}, id='d096170c-6617-4540-a1c9-66bda89e6b19'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 63, 'prompt_tokens': 303, 'total_tokens': 366, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_75546bd1a7', 'id': 'chatcmpl-D5toF6D98gQF1oZRNM4pK7Wf4B1rV', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019c2e03-f782-7711-9021-b4170874d305-0', tool_calls=[{'name': 'retriever_tool', 'args': {'query': '겨울왕국 엘사 예쁘다'}, 'id': 'call_OlxccDmCmNJHYW1UtMGma726', 'type': 'tool_call'}, {'name': 'retriever_tool', 'args': {'query': '백설